# API

The project ships a small [Quart](https://palletsprojects.com/projects/quart/) API with [quart-schema](https://quart-schema.readthedocs.io/) validation. Use the [Services](../../srv/services.qmd) page to open the site; this notebook calls the API **from inside the container** on **`http://127.0.0.1:5000`** (the same port Supervisor uses — see `build/conf/supervisor/conf.d/api.conf`).

In [1]:
#| include: false
import json
import os

import httpx

os.chdir("/content")

BASE = "http://127.0.0.1:5000"

## Overview

Routes live in `api/src/api/__init__.py`. Request and response models use **Pydantic** `BaseModel` classes so [quart-schema](https://quart-schema.readthedocs.io/) can validate and serialize JSON (including `datetime` fields). Example endpoints:

* `GET /healthcheck` — JSON status
* `POST /echo` — echoes JSON body
* `POST /todos/` — validates a `TodoIn` payload and returns a `TodoOut`

## Live requests

The cells below require the **api** Supervisor program to be running (`katx status`).

In [2]:
with httpx.Client(base_url=BASE, timeout=10.0) as client:
    r = client.get("/healthcheck")
print(r.status_code)
print(r.json())

200
{'status': 'alive'}


In [3]:
payload = {"hello": "katapult"}
with httpx.Client(base_url=BASE, timeout=10.0) as client:
    r = client.post("/echo", json=payload)
print(r.status_code)
print(json.dumps(r.json(), indent=2))

200
{
  "extra": true,
  "input": {
    "hello": "katapult"
  }
}


In [4]:
from datetime import datetime, timezone

todo = {"task": "Write docs", "due": datetime.now(timezone.utc).isoformat()}
with httpx.Client(base_url=BASE, timeout=10.0) as client:
    r = client.post("/todos/", json=todo)
print(r.status_code)
ct = r.headers.get("content-type", "")
if "application/json" in ct and r.content:
    print(json.dumps(r.json(), indent=2))
else:
    print(r.text or "(empty body)")

200
{
  "due": "2026-04-02T22:40:53.409007Z",
  "id": 1,
  "task": "Write docs"
}


In [5]:
bad = {"task": 123}  # wrong type for quart-schema validation
with httpx.Client(base_url=BASE, timeout=10.0) as client:
    r = client.post("/todos/", json=bad)
print(r.status_code)
print(r.text)

400
<!doctype html>
<html lang=en>
<title>400 Bad Request</title>
<h1>Bad Request</h1>
<p>The browser (or proxy) sent a request that this server could not understand.</p>



## Where files are saved

Package layout under `api/`:

In [6]:
#| label: fig-api-tree
#| echo: false
#| fig-cap: "api package layout"
!tree -nF api -L 3 -I '__pycache__|*.egg-info'

api/
├── README.md
├── pyproject.toml
├── src/
│   └── api/
│       └── __init__.py
└── uv.lock

3 directories, 4 files


## Dependencies

From a shell in the container (`katx connect`):

```bash
cd /content/api
uv add <package>
```

## Production layout

* Supervisor runs `uv run api` in `/content/api`.
* Nginx exposes the app at **`/<project_slug>/api`** and strips that prefix before proxying to port **5000**.

## Turn off the API service

Remove or rename `build/conf/supervisor/conf.d/api.conf`, then **`katx build`** and **`katx up`**.